# 03 — Multimodal Fusion + Evaluation

**Tham chiếu:** `docs/FinalTerm/pipeline_tong_the.md` Phase 4–7 · `docs/multimodal_fusion_architecture.md`.

Notebook này thực hiện:

| Phần | Mô tả | GPU? |
|---|---|---|
| 1 | Smoke tests các architecture variants | nhẹ |
| 2 | Train **Full Fusion (Concat)** — 5-fold CV (main result) | ✅ |
| 3 | Train **Cross-Attention Fusion** — 5-fold CV (RQ4) | ✅ |
| 4 | Ablation studies — prototype 70/15/15 từng variant | ✅ |
| 5 | Tổng hợp predictions + bảng so sánh tất cả model | CPU |
| 6 | **Discordance subgroup analysis** (RQ3 ⭐ Novelty) | CPU |
| 7 | Multi-task metric (Head 2 Echogenicity — RQ2) | CPU |
| 8 | Bảng cuối + trả lời 4 RQ | CPU |
| 9 | Grad-CAM visualization (Phase 7 stretch) | nhẹ |

**Trên Colab T4:** Full Fusion 5-fold ≈ 3–4 giờ · Cross-Attn 5-fold ≈ 3–4 giờ · 4 ablation prototype ≈ 2–3 giờ. Phần phân tích (5–8) chỉ cần CPU sau khi có OOF predictions.

**Cách dùng pragmatic:**
1. Chạy Phần 1 (smoke) trên local hoặc Colab — verify code.
2. Chạy Phần 2 Full Fusion 5-fold trên Colab (3–4h) — đây là main result, bắt buộc.
3. Phần 3 + 4 chạy thêm trên Colab nếu thời gian cho phép.
4. Phần 5–8 (analysis) chạy local sau khi đã download OOF CSV về.

## 0. Setup — Local & Google Colab

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/it2039-xulytinhieuhinhanhykhoa/project')
    assert PROJECT_ROOT.exists(), f'Không thấy project tại {PROJECT_ROOT}'
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('IN_COLAB     =', IN_COLAB)

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader

from src.clinical_rules import annotate_dataframe, ECHOGENICITY_CLASSES
from src.dataset import (
    CarotidDataset, TABULAR_FEATURES, build_transforms, carotid_collate,
    stratified_split,
)
from src.models import MultimodalFusionModel
from src.utils import (
    CSV_PATH, CHECKPOINTS_DIR, FIGURES_DIR, RESULTS_DIR,
    get_device, set_seed,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

SEED = 42
N_SPLITS = 5
set_seed(SEED)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

device = get_device()
print('Device:', device)
if device.type == 'cpu':
    print('⚠️  Không có CUDA — training cells sẽ skip. Chạy Colab cho training, local cho analysis.')

# Load annotated dataset
annotated_csv = RESULTS_DIR / 'carotid_annotated.csv'
if annotated_csv.exists():
    df = pd.read_csv(annotated_csv)
else:
    df = annotate_dataframe(pd.read_csv(CSV_PATH))
    df.to_csv(annotated_csv, index=False)
assert int(df['is_discordant'].sum()) == 33
print(f'df {df.shape}, 33 discordant ✓')

## 1. Smoke test — 3 architecture variants

Verify trước khi train: Full Fusion (Concat + projection) · Cross-Attention · w/o projection ablation. Tất cả phải có cùng output shape signature.

In [ ]:
variants = [
    {'name': 'Full Fusion (Concat)',     'cnn_use_projection': True,  'fusion_type': 'concat'},
    {'name': 'Cross-Attention Fusion',   'cnn_use_projection': True,  'fusion_type': 'cross_attn'},
    {'name': 'w/o projection ablation',  'cnn_use_projection': False, 'fusion_type': 'concat'},
]
imgs = torch.randn(6, 3, 224, 224)
n_imgs = torch.tensor([1, 5], dtype=torch.long)
tab = torch.randn(2, 9)

for v in variants:
    name = v.pop('name')
    m = MultimodalFusionModel(pretrained=False, **v)
    out = m(imgs, n_imgs, tab)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:35s} · CNN dim={m.cnn_branch.output_dim:>4} · params={n_params/1e6:.2f}M · plaque={tuple(out["plaque_logits"].shape)}')

## 2. Training function — multi-task fusion

Hàm dùng chung cho mọi variant (Full / Cross-Attn / ablations).

**Multi-task loss:** `1.0·L_plaque + 0.5·L_echogenicity + 0.3·L_reclassify`
- Head 1 (Plaque, 2-class): Weighted CE
- Head 2 (Echogenicity, 4-class): Weighted CE — imbalance None=205 vs Low=28/Inter=40/High=27
- Head 3 (Reclassify ESC/EAS, binary): BCE-with-logits

**3-stage finetune:** freeze backbone (5ep) → unfreeze layer4 (15ep) → unfreeze all + lr/10 (10ep) — đồng nhất với CNN-only baseline trong 02.

In [ ]:
FUSION_CFG = dict(
    seed=SEED, batch_size=16, weight_decay=1e-4, gradient_clip=1.0,
    lr_backbone=1e-5, lr_new_layers=1e-4,
    stage1_epochs=5, stage2_epochs=15, stage3_epochs=10,
    lambda_plaque=1.0, lambda_echo=0.5, lambda_reclassify=0.3,
    image_size=224,
)


def transform_tabular(df_in, scaler, *, zero_features: tuple = ()):
    """Encode Sex, apply scaler, optionally zero-out certain features (for ablation).
    Returns float32 array shape [N, 9]."""
    from src.dataset import encode_sex
    x = df_in[TABULAR_FEATURES].copy()
    x['Sex'] = encode_sex(x['Sex'])
    arr = scaler.transform(x.to_numpy(dtype=np.float32))
    for feat in zero_features:
        if feat in TABULAR_FEATURES:
            arr[:, TABULAR_FEATURES.index(feat)] = 0.0
    return arr.astype(np.float32)


def make_loader(df_part, *, train: bool, batch_size: int, image_size: int):
    ds = CarotidDataset(df_part, image_transform=build_transforms(train=train, image_size=image_size))
    return DataLoader(ds, batch_size=batch_size, shuffle=train,
                       collate_fn=carotid_collate, num_workers=0, pin_memory=True)


def train_fusion_one_fold(
    df_tr, df_va, *, cfg, device, fold_label='proto',
    cnn_use_projection: bool = True, fusion_type: str = 'concat',
    zero_features: tuple = (),
):
    """Train MultimodalFusionModel trên 1 fold. Trả về dict:
       model, history, val_preds (DataFrame mọi head + discordant mask), scaler.
    """
    set_seed(cfg['seed'])

    # Scaler fit trên train (raw values, sau khi encode Sex)
    from src.dataset import encode_sex
    X_tr = df_tr[TABULAR_FEATURES].copy()
    X_tr['Sex'] = encode_sex(X_tr['Sex'])
    scaler = StandardScaler().fit(X_tr.to_numpy(dtype=np.float32))

    tr_loader = make_loader(df_tr, train=True,  batch_size=cfg['batch_size'], image_size=cfg['image_size'])
    va_loader = make_loader(df_va, train=False, batch_size=cfg['batch_size'], image_size=cfg['image_size'])

    if zero_features:
        # Ablation: pre-scale + zero-out trong _X_raw, để dataset không scale lại.
        tr_loader.dataset._X_raw = transform_tabular(df_tr, scaler, zero_features=zero_features)
        va_loader.dataset._X_raw = transform_tabular(df_va, scaler, zero_features=zero_features)
        tr_loader.dataset.scaler = None
        va_loader.dataset.scaler = None
    else:
        # Path mặc định: _X_raw raw, dataset.scaler sẽ scale on-the-fly.
        tr_loader.dataset.scaler = scaler
        va_loader.dataset.scaler = scaler

    # Model
    model = MultimodalFusionModel(
        pretrained=True,
        cnn_use_projection=cnn_use_projection,
        fusion_type=fusion_type,
    ).to(device)

    # Class weights (compute_class_weight)
    y_pl = df_tr['Plaque_present'].values
    w_pl = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_pl)
    crit_plaque = nn.CrossEntropyLoss(weight=torch.tensor(w_pl, dtype=torch.float32, device=device))

    echo_idx_map = {c: i for i, c in enumerate(ECHOGENICITY_CLASSES)}
    y_echo = df_tr['Plaque_echogenicity'].map(echo_idx_map).values
    w_echo = compute_class_weight('balanced', classes=np.arange(4), y=y_echo)
    crit_echo = nn.CrossEntropyLoss(weight=torch.tensor(w_echo, dtype=torch.float32, device=device))

    crit_reclass = nn.BCEWithLogitsLoss()

    def make_opt(lr_bb, lr_new):
        groups = [{'params': model.cnn_branch.backbone.parameters(), 'lr': lr_bb}]
        if model.cnn_branch.use_projection:
            groups.append({'params': model.cnn_branch.projection.parameters(), 'lr': lr_new})
        groups += [
            {'params': model.mlp_branch.parameters(), 'lr': lr_new},
            {'params': model.fusion.parameters(),     'lr': lr_new},
            {'params': model.heads.parameters(),      'lr': lr_new},
        ]
        return torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])

    history = []
    best_auc, best_state = -1.0, None

    def run_epoch(loader, train: bool, opt=None):
        model.train(train)
        total_loss, ys_pl, ps_pl, ys_echo, ps_echo, ys_rc, ps_rc, discs = [], [], [], [], [], [], [], []
        for batch in loader:
            imgs = batch['images'].to(device, non_blocking=True)
            n_imgs = batch['n_images'].to(device)
            tab = batch['tabular'].to(device)
            y_p = batch['plaque'].to(device)
            y_e = batch['echo'].to(device)
            y_r = batch['reclassify'].to(device)

            with torch.set_grad_enabled(train):
                out = model(imgs, n_imgs, tab)
                Lp = crit_plaque(out['plaque_logits'], y_p)
                Le = crit_echo(out['echo_logits'], y_e)
                Lr = crit_reclass(out['reclassify_logit'], y_r)
                L = cfg['lambda_plaque']*Lp + cfg['lambda_echo']*Le + cfg['lambda_reclassify']*Lr
                if train:
                    opt.zero_grad(); L.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
                    opt.step()
            total_loss.append(L.item())
            ys_pl.append(y_p.detach().cpu().numpy())
            ps_pl.append(torch.softmax(out['plaque_logits'], dim=1)[:, 1].detach().cpu().numpy())
            ys_echo.append(y_e.detach().cpu().numpy())
            ps_echo.append(out['echo_logits'].argmax(dim=1).detach().cpu().numpy())
            ys_rc.append(y_r.detach().cpu().numpy())
            ps_rc.append(torch.sigmoid(out['reclassify_logit']).detach().cpu().numpy())
            discs.append(batch['discordant'].numpy())
        return (float(np.mean(total_loss)),
                np.concatenate(ys_pl), np.concatenate(ps_pl),
                np.concatenate(ys_echo), np.concatenate(ps_echo),
                np.concatenate(ys_rc), np.concatenate(ps_rc),
                np.concatenate(discs))

    # 3-stage finetune
    stages = [
        (1, cfg['stage1_epochs'], lambda: model.cnn_branch.freeze_backbone(), cfg['lr_backbone'],    cfg['lr_new_layers']),
        (2, cfg['stage2_epochs'], lambda: model.cnn_branch.unfreeze_layer4(), cfg['lr_backbone'],    cfg['lr_new_layers']),
        (3, cfg['stage3_epochs'], lambda: model.cnn_branch.unfreeze_all(),    cfg['lr_backbone']/10, cfg['lr_new_layers']/10),
    ]
    for s_id, n_ep, freeze_fn, lr_bb, lr_new in stages:
        print(f'[{fold_label}] Stage {s_id} — {n_ep} epochs (lr_bb={lr_bb:.0e}, lr_new={lr_new:.0e})')
        freeze_fn()
        opt = make_opt(lr_bb, lr_new)
        for ep in range(1, n_ep + 1):
            tr_L, *_ = run_epoch(tr_loader, train=True, opt=opt)
            va_L, va_yp, va_pp, va_ye, va_pe, va_yr, va_pr, va_disc = run_epoch(va_loader, train=False)
            auc = roc_auc_score(va_yp, va_pp) if len(set(va_yp)) > 1 else float('nan')
            f1 = f1_score(va_yp, (va_pp >= 0.5).astype(int), zero_division=0)
            echo_f1 = f1_score(va_ye, va_pe, average='macro', zero_division=0)
            history.append({'stage': s_id, 'epoch': ep, 'tr_loss': tr_L, 'va_loss': va_L,
                            'va_auc': auc, 'va_f1': f1, 'va_echo_f1': echo_f1})
            print(f'  S{s_id} ep{ep:02d}: tr_L={tr_L:.3f}  va_L={va_L:.3f}  '
                  f'AUC={auc:.3f}  F1={f1:.3f}  EchoF1={echo_f1:.3f}')
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    _, va_yp, va_pp, va_ye, va_pe, va_yr, va_pr, va_disc = run_epoch(va_loader, train=False)
    val_preds = pd.DataFrame({
        'Patient_ID': df_va['Patient_ID'].values,
        'y_true': va_yp,
        'y_prob': va_pp,
        'y_pred': (va_pp >= 0.5).astype(int),
        'echo_true': va_ye,
        'echo_pred': va_pe,
        'reclassify_true': va_yr,
        'reclassify_prob': va_pr,
        'is_discordant': va_disc,
    })

    return {'model': model, 'history': pd.DataFrame(history), 'val_preds': val_preds,
            'best_auc': best_auc, 'scaler': scaler}

In [ ]:
def metrics_full(y_true, y_prob, discordant_mask, threshold=0.5, name=''):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')
    dm = np.asarray(discordant_mask, dtype=bool); n_disc = int(dm.sum())
    if n_disc > 0:
        y_d, p_d = y_true[dm], y_prob[dm]
        yhat_d = (p_d >= threshold).astype(int)
        d_tn, d_fp, d_fn, d_tp = confusion_matrix(y_d, yhat_d, labels=[0, 1]).ravel()
        d_sens = d_tp / (d_tp + d_fn) if (d_tp + d_fn) else 0.0
        d_npv  = d_tn / (d_tn + d_fn) if (d_tn + d_fn) else 0.0
    else:
        d_sens = d_npv = float('nan')
    return {'model': name, 'AUC': auc, 'F1': f1_score(y_true, y_pred, zero_division=0),
            'Sens': sens, 'Spec': spec, 'Sens@discordant': d_sens,
            'NPV@discordant': d_npv, 'n_discordant': n_disc}


def summarize_folds(fm_df, name):
    out = {'model': name}
    for c in ['AUC', 'F1', 'Sens', 'Spec', 'Sens@discordant', 'NPV@discordant']:
        m, s = fm_df[c].mean(), fm_df[c].std()
        out[c] = f'{m:.3f} ± {s:.3f}'
    out['n_disc_total'] = int(fm_df['n_discordant'].sum())
    return out

## 3. Full Fusion (Concat) — 5-fold CV (main result)

Đây là kết quả chính của đề tài. Output: `pred_fusion_concat.csv` với OOF predictions cho cả 3 task.

In [ ]:
def run_kfold(*, name, cnn_use_projection=True, fusion_type='concat', zero_features=(),
              save_preds=True, save_ckpt=True):
    """Stratified 5-fold CV. Lưu OOF predictions + checkpoints."""
    if device.type == 'cpu':
        print(f'[{name}] Skip — no GPU.')
        return None, None
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    y_strat = df['Plaque_present'].values
    oof_rows = []
    fold_metrics = []

    for k, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df)), y_strat), start=1):
        print(f'\n========== {name} — FOLD {k}/{N_SPLITS} ==========')
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_va = df.iloc[va_idx].reset_index(drop=True)

        result = train_fusion_one_fold(
            df_tr, df_va, cfg=FUSION_CFG, device=device,
            fold_label=f'{name} fold{k}',
            cnn_use_projection=cnn_use_projection,
            fusion_type=fusion_type,
            zero_features=zero_features,
        )
        vp = result['val_preds'].copy()
        vp['fold'] = k
        oof_rows.append(vp)

        fm = metrics_full(vp['y_true'].values, vp['y_prob'].values, vp['is_discordant'].values,
                          name=f'{name} fold{k}')
        fm['fold'] = k
        fold_metrics.append(fm)
        print(f'  → AUC={fm["AUC"]:.3f}  F1={fm["F1"]:.3f}  Sens={fm["Sens"]:.3f}  '
              f'Sens@disc={fm["Sens@discordant"]:.3f} (n_disc={fm["n_discordant"]})')

        if save_ckpt:
            slug = name.lower().replace(' ', '_').replace('/', '_')
            torch.save(result['model'].state_dict(), CHECKPOINTS_DIR / f'{slug}_fold{k}.pt')

    oof = pd.concat(oof_rows, ignore_index=True)
    fm_df = pd.DataFrame(fold_metrics)
    if save_preds:
        slug = name.lower().replace(' ', '_').replace('/', '_')
        oof.to_csv(PREDICTIONS_DIR / f'pred_{slug}.csv', index=False)
        print(f'\nSaved OOF: pred_{slug}.csv')
    summary = summarize_folds(fm_df, name)
    print(f'\n[{name}] 5-fold summary:')
    print(pd.Series(summary).to_string())
    return oof, fm_df

In [ ]:
# === MAIN RESULT — Full Fusion Concat 5-fold ===
# Trên Colab T4 ≈ 3–4 giờ (≈ 30–40 phút/fold)
oof_full, fm_full = run_kfold(name='Fusion Concat', cnn_use_projection=True, fusion_type='concat')

## 4. Cross-Attention Fusion — 5-fold (RQ4)

So sánh với Concat trong cùng điều kiện. Skip nếu thời gian không đủ — Concat result vẫn đủ trả lời RQ1/3.

In [ ]:
oof_xattn, fm_xattn = run_kfold(name='Fusion CrossAttn', cnn_use_projection=True, fusion_type='cross_attn')

## 5. Ablation studies (prototype 70/15/15)

Single-fold để chứng minh đóng góp của từng thành phần. Mỗi variant ≈ 30–40 phút trên T4. Chạy 5-fold full nếu thời gian cho phép.

In [ ]:
def run_prototype(*, name, cnn_use_projection=True, fusion_type='concat', zero_features=()):
    """Train 1 fold 70/15/15 cho ablation prototype. Trả về dict metrics."""
    if device.type == 'cpu':
        print(f'[{name}] Skip — no GPU.'); return None
    df_tr, df_va, df_te = stratified_split(df, test_size=0.15, val_size=0.15, seed=SEED)
    print(f'[{name}] train/val/test = {len(df_tr)}/{len(df_va)}/{len(df_te)}')

    result = train_fusion_one_fold(
        df_tr, df_va, cfg=FUSION_CFG, device=device, fold_label=name,
        cnn_use_projection=cnn_use_projection, fusion_type=fusion_type,
        zero_features=zero_features,
    )
    vp = result['val_preds']
    m = metrics_full(vp['y_true'].values, vp['y_prob'].values, vp['is_discordant'].values, name=name)
    print(f'  → {name}: AUC={m["AUC"]:.3f}  F1={m["F1"]:.3f}  '
          f'Sens@disc={m["Sens@discordant"]:.3f} (n={m["n_discordant"]})')
    return m

In [ ]:
ablation_results = []
ablation_specs = [
    {'name': 'Full',              'cnn_use_projection': True,  'fusion_type': 'concat', 'zero_features': ()},
    {'name': 'w/o Lp(a)',         'cnn_use_projection': True,  'fusion_type': 'concat', 'zero_features': ('Lp(a)_mg_dL',)},
    {'name': 'w/o ApoB',          'cnn_use_projection': True,  'fusion_type': 'concat', 'zero_features': ('ApoB_mg_dL',)},
    {'name': 'w/o IMT_mm',        'cnn_use_projection': True,  'fusion_type': 'concat', 'zero_features': ('IMT_mm',)},
    {'name': 'w/o CNN projection','cnn_use_projection': False, 'fusion_type': 'concat', 'zero_features': ()},
]
for spec in ablation_specs:
    m = run_prototype(**spec)
    if m is not None:
        ablation_results.append(m)

ablation_df = pd.DataFrame(ablation_results) if ablation_results else None
if ablation_df is not None:
    ablation_df.to_csv(RESULTS_DIR / 'ablation_summary.csv', index=False)
    print('\nAblation summary:')
    print(ablation_df[['model', 'AUC', 'F1', 'Sens', 'Spec', 'Sens@discordant']].round(3).to_string(index=False))

## 6. Tổng hợp — Fusion vs Baselines

Load OOF predictions của 5 baseline (đã sinh ở 02_baseline) + Fusion → bảng so sánh + plot.

In [ ]:
def load_pred_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else None

preds = {
    'ESC/EAS Rule': load_pred_csv(PREDICTIONS_DIR / 'pred_esceas_rule.csv'),
    'RF':           load_pred_csv(PREDICTIONS_DIR / 'pred_rf.csv'),
    'XGBoost':      load_pred_csv(PREDICTIONS_DIR / 'pred_xgb.csv'),
    'MLP':          load_pred_csv(PREDICTIONS_DIR / 'pred_mlp.csv'),
    'CNN-only':     load_pred_csv(PREDICTIONS_DIR / 'pred_cnn_only.csv'),
    'Fusion Concat': load_pred_csv(PREDICTIONS_DIR / 'pred_fusion_concat.csv'),
    'Fusion CrossAttn': load_pred_csv(PREDICTIONS_DIR / 'pred_fusion_crossattn.csv'),
}
preds = {k: v for k, v in preds.items() if v is not None}
print('Loaded predictions:', list(preds.keys()))

In [ ]:
# Tính metrics overall cho từng model (OOF gộp 5 fold = predictions trên toàn 300)
comparison_rows = []
for name, pp in preds.items():
    m = metrics_full(pp['y_true'].values, pp['y_prob'].values, pp['is_discordant'].values, name=name)
    comparison_rows.append(m)

cmp_df = pd.DataFrame(comparison_rows)
cmp_df_view = cmp_df[['model', 'AUC', 'F1', 'Sens', 'Spec', 'Sens@discordant', 'NPV@discordant', 'n_discordant']]
cmp_df_view.to_csv(RESULTS_DIR / 'final_comparison.csv', index=False)
cmp_df_view.round(3)

In [ ]:
# Plot — AUC + Sens@discordant
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
models = cmp_df['model'].tolist()
colors_map = {
    'ESC/EAS Rule': '#888888',
    'RF': '#4c72b0', 'XGBoost': '#2e6094', 'MLP': '#7a9cc4',
    'CNN-only': '#55a868',
    'Fusion Concat': '#dd8452', 'Fusion CrossAttn': '#b14d2a',
}
colors = [colors_map.get(m, '#888') for m in models]

axes[0].bar(models, cmp_df['AUC'].fillna(0), color=colors)
for i, v in enumerate(cmp_df['AUC'].fillna(0)):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
axes[0].set_ylim(0, 1.05); axes[0].set_title('AUC-ROC (overall, OOF)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(models, cmp_df['Sens@discordant'].fillna(0), color=colors)
for i, v in enumerate(cmp_df['Sens@discordant'].fillna(0)):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Sensitivity @ discordant (NOVELTY — 33 cases)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_final_comparison.png')
plt.show()

## 7. Discordance subgroup analysis (RQ3 ⭐ Novelty)

Đây là phần đóng góp khoa học cốt lõi của đề tài. So sánh trên **33 discordant cases**:
- MLP-only miss bao nhiêu? (vì lipid panel "ổn")
- CNN-only miss bao nhiêu? (vì plaque khó nhìn)
- Fusion bắt được bao nhiêu thêm so với từng nhánh đơn?

In [ ]:
# Bảng case-by-case: dòng = patient discordant (Plaque=1), cột = predictions của các model
discordant_patients = df[df['is_discordant']]['Patient_ID'].tolist()
case_table = pd.DataFrame({'Patient_ID': discordant_patients})
case_table = case_table.merge(
    df[['Patient_ID', 'Plaque_present', 'Lp(a)_mg_dL', 'LDL_C_mg_dL',
        'Plaque_echogenicity', 'discordance_subtype']],
    on='Patient_ID', how='left'
)

for name, pp in preds.items():
    sub = pp.set_index('Patient_ID').loc[discordant_patients]
    case_table[f'{name}_pred'] = sub['y_pred'].values
    case_table[f'{name}_prob'] = sub['y_prob'].round(3).values

case_table.to_csv(RESULTS_DIR / 'discordance_case_table.csv', index=False)
case_table.head(10)

In [ ]:
# Số case discordant POSITIVE (plaque=1) mà mỗi model bắt được
disc_pos = case_table[case_table['Plaque_present'] == 1]
n_disc_pos = len(disc_pos)
print(f'Discordant cases với plaque=1: {n_disc_pos} / 33')
print()
rows = []
for name in preds.keys():
    pred_col = f'{name}_pred'
    if pred_col in disc_pos.columns:
        caught = int((disc_pos[pred_col] == 1).sum())
        rows.append({'model': name, 'caught': caught, 'missed': n_disc_pos - caught,
                     'recall': caught / n_disc_pos if n_disc_pos else 0.0})
catch_df = pd.DataFrame(rows).sort_values('recall', ascending=False)
catch_df['recall_pct'] = (catch_df['recall'] * 100).round(1).astype(str) + '%'
catch_df

In [ ]:
# Venn-like analysis: cases mà Fusion bắt được nhưng MLP/CNN-only miss
if 'Fusion Concat' in preds and 'MLP' in preds and 'CNN-only' in preds:
    fusion_catches = disc_pos[disc_pos['Fusion Concat_pred'] == 1]['Patient_ID']
    mlp_catches    = disc_pos[disc_pos['MLP_pred'] == 1]['Patient_ID']
    cnn_catches    = disc_pos[disc_pos['CNN-only_pred'] == 1]['Patient_ID']

    fusion_only = set(fusion_catches) - (set(mlp_catches) | set(cnn_catches))
    print(f'Cases Fusion bắt được mà CẢ MLP-only và CNN-only đều miss: {len(fusion_only)}')
    if fusion_only:
        print(case_table[case_table['Patient_ID'].isin(fusion_only)]
              [['Patient_ID', 'Lp(a)_mg_dL', 'LDL_C_mg_dL', 'Plaque_echogenicity',
                'discordance_subtype']].to_string(index=False))
        print('\n→ Đây là giá trị lâm sàng định lượng được của Fusion (RQ3).')
else:
    print('(Cần ít nhất pred_fusion_concat.csv, pred_mlp.csv, pred_cnn_only.csv)')

In [ ]:
# Plot — bar chart recall on discordant positive cases
if len(catch_df):
    fig, ax = plt.subplots(figsize=(10, 4))
    bar_colors = [colors_map.get(m, '#888') for m in catch_df['model']]
    ax.bar(catch_df['model'], catch_df['recall'], color=bar_colors)
    for i, (n, r) in enumerate(zip(catch_df['caught'], catch_df['recall'])):
        ax.text(i, r + 0.02, f'{n}/{n_disc_pos}\n({r*100:.0f}%)', ha='center', fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel(f'Recall on discordant+plaque (n={n_disc_pos})')
    ax.set_title('Mỗi model bắt được bao nhiêu discordant cases có plaque thật?')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_discordance_recall.png')
    plt.show()

## 8. RQ2 — Echogenicity 4-class accuracy (Multi-task)

Lp(a) + ApoB có giúp Head 2 (echogenicity classification) tốt hơn CNN-only không? So sánh Head 2 F1-macro của Full Fusion vs ablation `w/o Lp(a)` / `w/o ApoB`.

In [ ]:
if 'Fusion Concat' in preds and 'echo_pred' in preds['Fusion Concat'].columns:
    fc = preds['Fusion Concat']
    echo_f1 = f1_score(fc['echo_true'], fc['echo_pred'], average='macro', zero_division=0)
    echo_acc = accuracy_score(fc['echo_true'], fc['echo_pred'])
    print(f'Fusion Concat — Head 2 (Echogenicity):')
    print(f'  F1-macro = {echo_f1:.3f}')
    print(f'  Accuracy = {echo_acc:.3f}')
    print()
    print('Per-class report:')
    print(classification_report(fc['echo_true'], fc['echo_pred'],
                                target_names=list(ECHOGENICITY_CLASSES), zero_division=0))
    # Confusion matrix
    cm = confusion_matrix(fc['echo_true'], fc['echo_pred'], labels=range(4))
    fig, ax = plt.subplots(figsize=(5, 4.2))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=ECHOGENICITY_CLASSES, yticklabels=ECHOGENICITY_CLASSES, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Head 2 — Echogenicity confusion matrix (F1-macro={echo_f1:.3f})')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_echogenicity_confusion.png')
    plt.show()
else:
    print('(Chưa có Fusion predictions với echo_pred — chạy Phần 3 trước)')

## 9. Trả lời 4 Research Questions

In [ ]:
rq_answers = []

# RQ1: Fusion AUC > CNN-only + MLP-only?
if 'Fusion Concat' in preds:
    auc_fusion = cmp_df.loc[cmp_df['model'] == 'Fusion Concat', 'AUC'].iloc[0]
    auc_cnn    = cmp_df.loc[cmp_df['model'] == 'CNN-only',     'AUC'].iloc[0] if 'CNN-only' in preds else np.nan
    auc_mlp    = cmp_df.loc[cmp_df['model'] == 'MLP',          'AUC'].iloc[0] if 'MLP' in preds else np.nan
    answer = 'YES' if (auc_fusion > max(auc_cnn or 0, auc_mlp or 0)) else 'NO'
    rq_answers.append({
        'RQ': 'RQ1', 'question': 'Fusion AUC > CNN-only & MLP-only?',
        'answer': answer,
        'evidence': f'Fusion={auc_fusion:.3f}, CNN-only={auc_cnn:.3f}, MLP={auc_mlp:.3f}',
    })

# RQ2: Lp(a)/ApoB cải thiện echogenicity?
if ablation_df is not None:
    # Cần Head 2 metric — trong ablation_df hiện chỉ có Head 1.
    # Báo cáo: nếu w/o Lp(a) hoặc w/o ApoB làm AUC giảm → suggestive cho RQ2.
    full_auc = ablation_df.loc[ablation_df['model'] == 'Full', 'AUC'].iloc[0] if (ablation_df['model'] == 'Full').any() else np.nan
    wo_lpa_auc = ablation_df.loc[ablation_df['model'] == 'w/o Lp(a)', 'AUC'].iloc[0] if (ablation_df['model'] == 'w/o Lp(a)').any() else np.nan
    wo_apob_auc = ablation_df.loc[ablation_df['model'] == 'w/o ApoB', 'AUC'].iloc[0] if (ablation_df['model'] == 'w/o ApoB').any() else np.nan
    rq_answers.append({
        'RQ': 'RQ2', 'question': 'Lp(a)/ApoB giúp echogenicity (Head 2)?',
        'answer': 'See ablation + Head 2 F1-macro của Phần 8',
        'evidence': f'Full AUC={full_auc:.3f}, w/o Lp(a)={wo_lpa_auc:.3f}, w/o ApoB={wo_apob_auc:.3f}',
    })

# RQ3: Fusion giải quyết Discordance?
if 'Fusion Concat' in preds:
    sens_fusion = cmp_df.loc[cmp_df['model'] == 'Fusion Concat', 'Sens@discordant'].iloc[0]
    sens_cnn    = cmp_df.loc[cmp_df['model'] == 'CNN-only',     'Sens@discordant'].iloc[0] if 'CNN-only' in preds else np.nan
    sens_mlp    = cmp_df.loc[cmp_df['model'] == 'MLP',          'Sens@discordant'].iloc[0] if 'MLP' in preds else np.nan
    sens_rule   = cmp_df.loc[cmp_df['model'] == 'ESC/EAS Rule', 'Sens@discordant'].iloc[0] if 'ESC/EAS Rule' in preds else np.nan
    answer = 'YES' if sens_fusion > max([s for s in [sens_cnn, sens_mlp, sens_rule] if not np.isnan(s)] + [0]) else 'NO'
    rq_answers.append({
        'RQ': 'RQ3 ⭐', 'question': 'Fusion solve Discordance > baselines?',
        'answer': answer,
        'evidence': f'Fusion={sens_fusion:.3f}, CNN={sens_cnn:.3f}, MLP={sens_mlp:.3f}, Rule={sens_rule:.3f}',
    })

# RQ4: Concat vs Cross-Attention?
if 'Fusion CrossAttn' in preds:
    auc_xattn = cmp_df.loc[cmp_df['model'] == 'Fusion CrossAttn', 'AUC'].iloc[0]
    auc_concat = cmp_df.loc[cmp_df['model'] == 'Fusion Concat', 'AUC'].iloc[0]
    winner = 'Cross-Attention' if auc_xattn > auc_concat else 'Concat'
    rq_answers.append({
        'RQ': 'RQ4', 'question': 'Concat vs Cross-Attention (n=300)?',
        'answer': winner,
        'evidence': f'Concat AUC={auc_concat:.3f}, CrossAttn AUC={auc_xattn:.3f}',
    })

rq_df = pd.DataFrame(rq_answers)
rq_df.to_csv(RESULTS_DIR / 'rq_answers.csv', index=False)
rq_df

## 10. Grad-CAM visualization (Phase 7 stretch)

Hiển thị vùng ảnh mà CNN branch chú ý nhất khi dự đoán plaque. Dùng `pytorch-grad-cam` (cài thêm nếu chạy).

In [ ]:
# Optional — cần cài: pip install grad-cam
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    GRAD_CAM_AVAILABLE = True
except ImportError:
    print('Cài: pip install grad-cam'); GRAD_CAM_AVAILABLE = False

if GRAD_CAM_AVAILABLE and device.type == 'cuda' and (CHECKPOINTS_DIR / 'fusion_concat_fold1.pt').exists():
    from PIL import Image
    from src.dataset import build_transforms
    from src.utils import IMAGES_DIR

    # Load fold 1 checkpoint
    model = MultimodalFusionModel(pretrained=False).to(device)
    model.load_state_dict(torch.load(CHECKPOINTS_DIR / 'fusion_concat_fold1.pt', map_location=device))
    model.eval()

    target_layer = model.cnn_branch.backbone.layer4[-1]
    cam = GradCAM(model=model.cnn_branch.backbone, target_layers=[target_layer])

    # Pick một case discordant với plaque=1 và echogenicity=Low (vulnerable)
    candidates = df[(df['is_discordant']) & (df['Plaque_present'] == 1) &
                    (df['Plaque_echogenicity'] == 'Low')]
    if len(candidates) == 0:
        candidates = df[df['Plaque_present'] == 1].head(1)
    pid = candidates['Patient_ID'].iloc[0]
    img_path = IMAGES_DIR / f'{pid}_CCA_L1.png'
    if not img_path.exists():
        img_path = IMAGES_DIR / f'{pid}_IMT.png'

    img = Image.open(img_path).convert('RGB')
    tf = build_transforms(train=False, image_size=224)
    input_tensor = tf(img).unsqueeze(0).to(device)
    grayscale_cam = cam(input_tensor=input_tensor)[0]
    img_np = np.array(img.resize((224, 224))).astype(np.float32) / 255.0
    cam_img = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(img_np); axes[0].set_title(f'{pid} ({img_path.name})'); axes[0].axis('off')
    axes[1].imshow(cam_img); axes[1].set_title('Grad-CAM (vùng CNN chú ý)'); axes[1].axis('off')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_gradcam.png')
    plt.show()
else:
    print('Skip Grad-CAM — cần cài grad-cam và có checkpoint fold 1.')

## Tổng kết Phase 4–7

**Saved predictions:**
- `results/predictions/pred_fusion_concat.csv` — Full Fusion 5-fold OOF (main result)
- `results/predictions/pred_fusion_crossattn.csv` — Cross-Attention 5-fold OOF (RQ4)

**Saved checkpoints:**
- `results/checkpoints/fusion_concat_fold{1..5}.pt`
- `results/checkpoints/fusion_crossattn_fold{1..5}.pt`

**Saved tables:**
- `results/final_comparison.csv` — bảng tổng hợp 7 model
- `results/discordance_case_table.csv` — 33 cases × predictions từng model
- `results/ablation_summary.csv` — kết quả 5 ablation variants
- `results/rq_answers.csv` — câu trả lời 4 RQ

**Saved figures (mới):**
- `fig_final_comparison.png` — AUC + Sens@discordant tất cả model
- `fig_discordance_recall.png` — recall trên discordant+plaque
- `fig_echogenicity_confusion.png` — confusion matrix Head 2
- `fig_gradcam.png` — visualize CNN attention (optional)

**Workflow Colab pragmatic:**
1. Upload `project/` lên Drive
2. Chạy Phần 1–2 verify
3. Chạy Phần 3 (Full Fusion 5-fold) — main result
4. Tùy thời gian: Phần 4 (Cross-Attn) + Phần 5 (ablations)
5. Phần 6–9 (analysis) chạy local sau khi đã sync OOF CSV về